# 第4章 数えるだけで漱石っぽくなる：bigramモデル

夏目漱石『こころ』で作る、はじめての言語モデル

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：『こころ』の「ある文字の次に来る文字」を数えて、漱石風（？）の文章を書くbigram言語モデル
- **身につく数学・理論**：データの前処理（文字コード変換・ルビ除去） ／ 正規表現の入門 ／ 条件付き確率 $P(\text{次} \mid \text{今})$ ／ bigram・n-gramモデルとマルコフ性 ／ データが足りない問題（スパース性）
- **新しく使うPython・PyTorch**：`urllib` でダウンロード ／ `bytes` の `decode` ／ `re.sub`（正規表現で置換） ／ `collections.Counter` ／ `zip` とスライス `text[1:]` ／ `random.choices` ／ ファイルの読み書き `with open`

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 60〜90分 |
| 難易度 | ★★☆☆☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> いよいよ、**自分の言語モデル**を作ります。材料は夏目漱石の『こころ』、道具は「数を数える」ことだけ。それなのに、完成したモデルは「私は先生の…」と、なんとなく漱石っぽい文章を書き始めます。この章のモデルが、これから第13章のGPTまで続く「進化の記録」のスタート地点です。

> **🔗 これまでの復習とつながり：第2章・第3章とのつながり**
>
> - **第2章**：コンピュータの中の文字は `bytes`（0〜255のバイト列）として保存され、文字コード（UTF-8など）で文字列に戻ること、そして文字をIDに対応づける語彙（`stoi`）を学びました。
> - **第3章**：言語モデルの仕事は「今の文脈の次に来るトークンの**確率分布**を出し、その確率に従って**サンプリング（くじ引き）**する」ことだと学びました。
>
> この章では、いよいよ既製品モデルを卒業し、小説1冊から「ある文字の次にどの文字が来るか」を数えて**自分で確率分布を作り、自分で文章をサンプリングする**最初の言語モデルを完成させます！

**ミナト**：言語モデルって、何億個もパラメータがあるすごいものじゃないんですか？ 数えるだけで作れるんですか？

**ボンゴ先生**：作れるとも。第3章で見たように、言語モデルの仕事は「次のトークンの確率を出す」ことだったね。だったら、**本当に数えてしまえばいい**。「『先』の次に何が来たか」を小説1冊ぶん全部数えれば、確率が出る。世界一素朴な言語モデルだ。素朴だからこそ、全部の部品が見える。

## 材料を手に入れる：青空文庫

**青空文庫**は、著作権の切れた文学作品を誰でも無料で読めるようにしている、ボランティアによる電子図書館です。夏目漱石（1916年没）の作品は誰でも自由に使えます。テキストファイルがGitHubにも置かれているので、そこからダウンロードしましょう。

新しいノートブック（`04-bigram.ipynb`）を作って、次を実行します。CPUのままでOKです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import urllib.request

url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/0984f7dc18a2270bdfd2ff99bf14fd1b3ebb4f65/"
       "cards/000148/files/773_ruby_5968/773_ruby_5968.txt")
try:
    with urllib.request.urlopen(url, timeout=30) as res:
        raw = res.read()
except Exception as e:
    raise RuntimeError(f"青空文庫の取得に失敗しました（ネット接続を確認し再実行するか付録Eを参照）: {e}") from e
print(type(raw), len(raw), "バイト")
print(raw[:40])

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
<class 'bytes'> 374152 バイト
b'\x82\xb1\x82\xb1\x82\xeb\r\n\x89\xc4\x96\xda\x9f\xf9\x90\xce\r\n\r\n--------------------'
```

</details>

ダウンロードできたのは **バイト列**（`bytes`）、つまり0〜255の数の並びです。まだ文字ではありません。第2章で学んだとおり、バイトを文字に戻すには「どの文字コードで書かれているか」を知る必要があります。

## 文字コードの壁を越える

いつものUTF-8で文字に戻してみると……

```python
text = raw.decode("utf-8")
```

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
UnicodeDecodeError: 'utf-8' codec can't decode byte 0x82 in position 0: invalid start byte
```

</details>

エラーです。「0番目のバイト `0x82` は、UTF-8としてありえない」と言われました。

> **📘 基本情報とのつながり：Shift_JIS と cp932**
>
> 青空文庫のファイルは、古くから日本のパソコンで使われてきた **Shift_JIS** という文字コードで書かれています。基本情報の文字コードの問題で出てくる、あれです。正確にはWindowsで拡張された版の **cp932**（Microsoftの文字コード表の932番）を使うと、機種依存文字まで正しく読めます。<br>
> `0x82 0xb1` はShift_JISで「こ」。`0x82 0xb1 0x82 0xb1 0x82 0xeb` で「こころ」です。上の出力の先頭と見比べてみてください。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
text = raw.decode("cp932")
print(text[:300])

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
こころ
夏目漱石

-------------------------------------------------------
【テキスト中に現れる記号について】

《》：ルビ
（例）私《わたくし》は

｜：ルビの付く文字列の始まりを特定する記号
（例）先生一人｜麦藁帽《むぎわらぼう》を

［＃］：入力者注　主に外字の説明や、傍点の位置の指定
　　　（数字は、JIS X 0213の面区点番号、または底本のページと行数）
（例）※［＃「てへん＋劣」、第3水準1-84-77］
--------------------------------------------
```

</details>

読めました！ でも、本文の前に説明書きがあり、本文にも `《わたくし》` のような**ルビ**（ふりがな）や `［＃…］` という注記が混ざっています。このまま学習させると、モデルは「私」の次に「《」が来る、と覚えてしまいます。漱石はそんな書き方をしていません。

<details><summary><b>基礎の補足：理解のための最小コード：`bytes.decode("cp932")` の入力・途中・出力</b>（クリックで開く）</summary>

| ステップ | 値（具体例） | 型 | 起きていること |
|---|---|---|---|
| 入力 | `b'\x82\xb1\x82\xb1\x82\xeb'` | `bytes`（長さ6） | 0〜255の生の数値列（2バイト×3文字ぶん） |
| 途中変換 | `(0x82,0xb1)→'こ'`, `(0x82,0xb1)→'こ'`, `(0x82,0xeb)→'ろ'` | cp932対応表で照合 | 保存時と同じ文字コード表で文字へ復号 |
| 出力 | `'こころ'` | `str`（長さ3） | Pythonで1文字ずつ数えられる文字列になる |

```python no-run
raw_small = b"\x82\xb1\x82\xb1\x82\xeb"
print(type(raw_small), len(raw_small))      # <class 'bytes'> 6
s = raw_small.decode("cp932")
print(type(s), len(s), s)                   # <class 'str'> 3 こころ
```

</details>

#### 📝 確認問題（`ch04-q01` · Lv.1 · bytes型と文字コード（UTF-8 vs cp932）の役割）

青空文庫からダウンロードした `raw` に対して `raw.decode("utf-8")` を実行すると `UnicodeDecodeError` になり、`raw.decode("cp932")` なら読める理由として正しいものはどれですか？

- **a.** `raw` は `cp932` で符号化されたバイト列であり、UTF-8規則だと先頭バイトが不正になるため
- **b.** `decode("cp932")` はファイル中のルビや注記を自動で削除する機能を持ち、UTF-8では記号を処理できず止まるため
- **c.** Python 3 の標準仕様は日本語のUTF-8に対応しておらず、日本語テキストを読むときは常に `cp932` を指定する必要があるため
- **d.** `raw` はダウンロード時点ですでに日本語の文字列 `str` になっているのに、誤って2回 `decode` を呼んでいるため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：文字コードは「文字とバイト列の対応表」です。保存された形式と読むときの形式が食い違うとどうなるでしょう？
- **ヒント 2**：青空文庫のファイルは Shift_JIS（Windows拡張 `cp932`）のバイト列として保存されているため、UTF-8の表では復号できません。
- **正解**：`a`

**考え方**：バイト列 `bytes` 自体はただの0〜255の数字の並びです。ファイルが保存されたときと同じ文字コード（青空文庫では `cp932`）を指定して `decode` することで、正しい文字列 `str` に復元できます。

1. `raw` の型は `bytes`（例：`b'\x82\xb1\x82\xb1\x82\xeb'`）。
1. UTF-8では `0x82` から始まる文字が存在しないため `UnicodeDecodeError`。
1. `cp932` では `0x82 0xb1`＝「こ」、`0x82 0xeb`＝「ろ」と対応するため、型 `str` の `'こころ'` に戻る。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html#ex-ch04-q01

</details>

## 掃除する：正規表現

機械学習では、**データの掃除（前処理）がとても大事**です。「ゴミを入れたらゴミが出てくる（Garbage In, Garbage Out）」という格言があるくらいです。ルビを消すには **正規表現**（regular expression）を使います。正規表現は「こういう形の文字列」というパターンを書くための小さな言語です。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import re

sample = "私《わたくし》はその人を常に先生《せんせい》と呼んでいた。"
print(re.sub(r"《.*?》", "", sample))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
私はその人を常に先生と呼んでいた。
```

</details>

#### 🐍 Python・PyTorchのポイント：正規表現の読み方

`re.sub(パターン, 置き換える文字, 対象)` は「パターンに合う部分を全部置き換える」関数です。`r"《.*?》"` は次のように読みます。

| 部分 | 意味 |
|---|---|
| `《` | 「《」という文字 |
| `.` | 何でもいい1文字 |
| `*` | 直前のものが0回以上くり返す |
| `?` | ただし、なるべく短く（控えめに） |
| `》` | 「》」という文字 |

つまり「《 から始まって、いちばん近い 》 で終わる部分」です。`?` を付け忘れると「なるべく長く」になって、最初の《から**最後の**》までをごっそり消してしまいます。試してみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
print(re.sub(r"《.*》", "", sample))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
私と呼んでいた。
```

</details>

「はその人を常に先生」まで消えてしまいました。正規表現は強力なぶん、ちょっとしたミスで大惨事になります。電動ノコギリみたいなものです。

<details><summary><b>基礎の補足：理解のための最小コード：`re.sub` の最短一致 `.*?` vs 最長一致 `.*`</b>（クリックで開く）</summary>

同じ入力 `s = "私《わたくし》は先生《せんせい》だ"`（`str`型）に対して、マッチする範囲（中間状態）と出力を比べてみましょう。

| パターン | マッチする区間（中間） | 置換後の出力（`str`型） | 理由 |
|---|---|---|---|
| `r"《.*?》"`（最短一致） | ①`《わたくし》`、②`《せんせい》` の2か所 | `"私は先生だ"` | `?` があるので一番近い `》` で止まる |
| `r"《.*》"`（最長一致） | `《わたくし》は先生《せんせい》` の1か所 | `"私だ"` | `?` がないので最後の `》` まで欲張って飲み込む |

```python no-run
import re
s = "私《わたくし》は先生《せんせい》だ"
print(re.sub(r"《.*?》", "", s))   # 私は先生だ
print(re.sub(r"《.*》", "", s))    # 私だ（間の「は先生」まで消滅！）
```

</details>

では、本文全体を掃除する関数を作ります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def clean_aozora(text):
    text = text.split("-------------------------------------------------------")[2]  # 冒頭の説明を除く
    text = text.split("底本：")[0]                   # 末尾の書誌情報を除く
    text = re.sub(r"《.*?》", "", text)              # ルビを除く
    text = re.sub(r"※?［＃.*?］", "", text)          # 入力者注を除く
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)           # 3行以上の空行を2行に
    return text.strip()

text = clean_aozora(raw.decode("cp932"))
print(len(text), "文字")
print(text[:150])

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
162467 文字
上　先生と私

一

　私はその人を常に先生と呼んでいた。だからここでもただ先生と書くだけで本名は打ち明けない。これは世間を憚かる遠慮というよりも、その方が私にとって自然だからである。私はその人の記憶を呼び起すごとに、すぐ「先生」といいたくなる。筆を執っても心持は同じ事である。よそよそしい頭文字など
```

</details>

16万字あまりのきれいな本文が手に入りました。文庫本1冊ぶんです。

> **🐍 Python・PyTorchのポイント：split と strip**
>
> `text.split("区切り")` は文字列を区切りで分割してリストにします。`[2]` で3番目（0から数えるので）を取り出しています。冒頭の説明は `-----` の線で挟まれているので、「線の前・線の間・線の後」の3つに分かれ、その3番目が本文です。<br>
> `strip()` は前後の空白や改行を取り除きます。

せっかく掃除したので、ファイルに保存しておきましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
with open("kokoro.txt", "w", encoding="utf-8") as f:
    f.write(text)
print("保存しました")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
保存しました
```

</details>

> **✨ コツ：Colabのファイルは消える**
>
> Colabの左側のフォルダのアイコンを押すと、`kokoro.txt` ができているのが見えます。ただし、このファイルは**接続が切れると消えます**。この教科書では各章の最初でダウンロードし直すので心配いりませんが、大事なファイルをずっと残したいときは、Googleドライブに保存します（第10章で、学習したモデルを保存するときにやります）。

#### 📝 確認問題（`ch04-q02` · Lv.2 · re.sub の最短一致 .*? と最長一致 .* の違い）

非貪欲（最短一致）の `?` の働きを確認します。次のコードを実行すると何が表示されますか？

```python
import re
s = "A《あ》とB《びー》"
print(re.sub(r"《.*?》", "", s))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：正規表現の `.*?`（非貪欲・最短一致）は、最初の `《` から一番近い `》` でマッチを止めます。
- **ヒント 2**：`《あ》` と `《びー》` がそれぞれ独立に空文字へ置換され、その間に挟まれた部分は残ります。
- **期待される出力**：`AとB`

**考え方**：`.*?` は最短一致（控えめなマッチ）なので、最初の `《` から最も近い `》` までを1つのまとまりとして置き換えます。そのため間の「とB」は残ります。

1. 1回目のマッチ：`《あ》` → 空文字 `""` に置換。
1. 間の `とB` はパターン外なのでそのまま残る。
1. 2回目のマッチ：`《びー》` → 空文字 `""` に置換。
1. 結果：`AとB`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html#ex-ch04-q02

</details>

#### 📝 確認問題（`ch04-q03` · Lv.2 · 前処理（ルビ・注記除去と split/strip）の必要性）

`clean_aozora` で本文の前後を切り落とし、ルビ `《…》` や入力者注 `［＃…］` を取り除いてから学習させる最も重要な理由はどれですか？

- **a.** Pythonの `Counter` は記号の `《` や `［` を数えようとすると必ず例外を発生させて停止する仕様があるため
- **b.** `split("-----")[2]` で本文を切り出さない限り、そもそも最初の `decode("cp932")` が実行できないため
- **c.** ルビと注記を取り除いて文字数を10分の1以下に減らさないと、Google Colabのメモリが不足して動かないため
- **d.** 注記やルビを残すと、モデルが「私」の次に「《」が来るなど本文にはない記号パターンを覚えてしまうため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：言語モデルは、渡された学習テキストに含まれる文字の並びをそのまま統計として学びます。
- **ヒント 2**：「ゴミを入れればゴミが出る（Garbage In, Garbage Out）」の観点で、ルビや底本情報が残る弊害を選びましょう。
- **正解**：`d`

**考え方**：データに混入した説明文・ルビ・注記はそのまま「次の文字の確率」をゆがめます。大規模LLMの事前学習でも、HTMLタグや広告文の除去などデータ前処理が性能を大きく左右します。

1. 生の青空文庫テキストには冒頭の説明、ルビ `《わたくし》`、末尾の底本情報が混ざっている。
1. そのまま数えると `私→《` や `底→本` のような漱石の文体と無関係なペアが学習される。
1. 前処理で本文だけを抽出することで、目的の分布（漱石の文章）を正しく学べる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html#ex-ch04-q03

</details>

## 文字を数える

まずは、どの文字がよく出てくるかを数えてみましょう。数えるのに便利な道具が `Counter` です。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
from collections import Counter

counts = Counter(text)
print("文字の種類：", len(counts))
print(counts.most_common(10))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
文字の種類： 2061
[('の', 7556), ('た', 6734), ('い', 6214), ('し', 5302), ('に', 4710), ('。', 4654), ('は', 4334), ('て', 4193), ('な', 4061), ('で', 3909)]
```

</details>

『こころ』は2061種類の文字でできていて、一番多いのは「の」でした。これが私たちの言語モデルの**語彙**（2061文字）になります。

<details><summary><b>基礎の補足：理解のための最小コード：`collections.Counter` の入力・中身・出力</b>（クリックで開く）</summary>

`Counter` は「要素 → 出現回数」を自動で集計する辞書（`dict` の仲間）です。普通の辞書との最大の違いは、**存在しないキーを聞いてもエラーにならず `0` を返す**点です。

| ステップ | コード / 値 | 型 | 意味 |
|---|---|---|---|
| 入力 | `s = "こころ"` | `str`（長さ3） | 1文字ずつ取り出せる文字列 |
| 集計後 | `c = Counter(s)` → `Counter({'こ': 2, 'ろ': 1})` | `Counter`（辞書の仲間） | 各文字の出現回数マップ |
| 種類数 | `len(c)` → `2` | `int` | ユニークな文字の種類（総文字数3ではない） |
| 上位取得 | `c.most_common(1)` → `[('こ', 2)]` | `list` of `(str, int)` | 多い順に `(文字, 回数)` タプルを並べたリスト |
| 未出現キー | `c["私"]` → `0` | `int` | 普通なら `KeyError` だが `Counter` は `0` |

```python no-run
from collections import Counter
c = Counter("こころ")
print(len(c), c["こ"], c["私"])   # 2 2 0
print(c.most_common(2))           # [('こ', 2), ('ろ', 1)]
```

</details>

#### 📝 確認問題（`ch04-q04` · Lv.2 · collections.Counter の集計・missing key=0・most_common）

`Counter` の特徴（存在しないキーへのアクセスと `len`）を確認します。次のコードを実行すると何が表示されますか？

```python
from collections import Counter
c = Counter("こころ")
print(len(c), c["こ"], c["私"])
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`Counter` の `len(c)` は総文字数ではなく「文字の種類数」、未登録キーへのアクセスはエラーではなく特定の整数を返します。
- **ヒント 2**：`"こころ"` は `'こ': 2, 'ろ': 1` の2種類で、未登録の `'私'` には `0` が返ります。
- **期待される出力**：`2 2 0`

**考え方**：`Counter("こころ")` は `{'こ': 2, 'ろ': 1}` となり、種類数 `len(c)` は `2`、`c["こ"]` は `2`、一度も出てこない `c["私"]` は `KeyError` ではなく `0` を返します。

1. `Counter("こころ")` → `Counter({'こ': 2, 'ろ': 1})`。
1. `len(c)` → キーの種類数なので `2`（文字数 `3` ではない点に注意）。
1. `c["こ"]` → `2`、`c["私"]` → 未登録キーなので `0`。
1. 出力：`2 2 0`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html#ex-ch04-q04

</details>

## 「次の文字」を数える

ここからが言語モデルです。「ある文字の次に、どの文字が来たか」の組（**ペア**）を全部数えます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
pairs = Counter(zip(text, text[1:]))
print(pairs.most_common(10))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
[(('た', '。'), 2960), (('し', 'た'), 2043), (('っ', 'た'), 1890), (('っ', 'て'), 1406), (('私', 'は'), 1352), (('し', 'て'), 1172), (('で', 'す'), 1142), (('て', 'い'), 1110), (('ま', 'し'), 1083), (('の', 'で'), 1071)]
```

</details>

#### 🐍 Python・PyTorchのポイント：zip とスライス

`text[1:]` は「1番目の文字から最後まで」、つまり先頭の1文字を除いた文字列です（**スライス**といいます）。`zip(text, text[1:])` は2つを横に並べて、「0番目と1番目」「1番目と2番目」「2番目と3番目」…という組を順に作ります。

| `text` | 私 | は | そ | の | 人 |
|---|---|---|---|---|---|
| `text[1:]` | は | そ | の | 人 | を |

縦に読むと「私→は」「は→そ」「そ→の」…。隣り合う2文字の組が全部手に入ります。

<details><summary><b>基礎の補足：理解のための最小コード：`zip(text, text[1:])` の入力・途中・出力</b>（クリックで開く）</summary>

| ステップ | 値（具体例） | 型・長さ | 起きていること |
|---|---|---|---|
| 入力 | `s = "こころの"` | `str`（長さ4） | 元の文字列 |
| 1文字ずらす | `s[1:]` → `"ころの"` | `str`（長さ3） | 先頭の1文字を切り落とした文字列 |
| `zip` で組にする | `list(zip(s, s[1:]))` → `[('こ','こ'), ('こ','ろ'), ('ろ','の')]` | `list` of `(str, str)`（長さ3） | 短い方の長さ `len(s) - 1` で止まり、隣接ペアができる |
| `Counter` で集計 | `Counter(zip(s, s[1:]))` → `{('こ','こ'):1, ('こ','ろ'):1, ('ろ','の'):1}` | `Counter` | 各2文字ペア `(a, b)` の回数がまとまる |

```python no-run
from collections import Counter
s = "こころの"
pairs_small = list(zip(s, s[1:]))
print(len(pairs_small), pairs_small)   # 3 [('こ', 'こ'), ('こ', 'ろ'), ('ろ', 'の')]
print(Counter(pairs_small))            # Counter({('こ', 'こ'): 1, ('こ', 'ろ'): 1, ('ろ', 'の'): 1})
```

</details>

「〜た。」で終わる文が2960回。さすが小説、過去形だらけです。では、「先」の次には何が来るでしょう？

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def next_counts(ch):
    c = Counter()
    for (a, b), n in pairs.items():   # 全部のペアの中から
        if a == ch:                   # 1文字目が ch のものだけ
            c[b] = n                  # 2文字目と回数を記録
    return c

c = next_counts("先")
print("「先」は全部で", sum(c.values()), "回")
print(c.most_common(5))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
「先」は全部で 678 回
[('生', 597), ('へ', 22), ('刻', 16), ('で', 8), ('を', 6)]
```

</details>

「先」678回のうち597回、つまり約88%が「先生」です。『こころ』は「先生」の物語ですからね。

#### 📐 数学のポイント：条件付き確率 $P(B \mid A)$：9ステップで理由から理解する

**① 解決したい具体的な問題**：「今の文字が『先』だ」と分かっているとき、次の文字が『生』『へ』『刻』…である確率を正しく出したい。

**② 素朴な方法（単なる出現回数・全体割合）ではなぜダメなのか？**
- もし文章全体の回数ランキング（**unigram**）だけを使うと、『こころ』で最多は「の」（7556回）なので、どんな文字の次でも「の」が一番選ばれやすくなり、「ののたのい…」のような日本語にならない文が出てしまいます。
- また、生の回数 `597` のままでは「678回中597回（88%）」なのか「1万回中597回（6%）」なのか分からず、合計1の確率分布にもなりません。

**③ 新しい発想**：小説全体を見るのではなく、**「1文字目が $a$ であるペアだけ」に世界を絞り込み**、その中で次が $b$ だった割合をとる。これが **条件付き確率** です。

**④ 記号の読み方表**：
| 記号 | 読み方 | 意味 |
|---|---|---|
| $P(B \mid A)$ | 「AのもとでのBの確率」 | 条件 $A$ が起きた世界で、出来事 $B$ が起きる割合 |
| $\mid$（縦棒） | 「〜という条件のもとで」 | **右側**が分かっている条件（分母）、**左側**が予測したいもの |
| $\text{count}(a, b)$ | 「$a$ の次に $b$ が来た回数」 | 隣接ペア `(a, b)` の出現回数 |
| $\text{count}(a)$ | 「$a$（の後に文字がある箇所）の回数」 | 条件 $a$ で始まるペアの総数 $\sum_b \text{count}(a, b)$ |

**⑤ 小さな10文字コーパスで手計算**：`s = "あいうあいうあいあえ"`（10文字・隣接9ペア）で、今の文字が「あ」のときを手計算してみましょう。
- 全9ペア：`あい, いう, うあ, あい, いう, うあ, あい, いあ, あえ`
- 「あ」で始まるペアだけ絞り込む：`あい`（3回）、`あえ`（1回）→ 合計4回
- したがって：
$$P(\text{い} \mid \text{あ}) = \frac{3}{4} = 0.75, \qquad P(\text{え} \mid \text{あ}) = \frac{1}{4} = 0.25$$
- 合計は $0.75 + 0.25 = 1.00$。きちんと確率分布の2条件（0以上・合計1）を満たします！

**⑥ 一般式**：『こころ』のような長い文章でも同じです。
$$P(b \mid a) = \frac{\text{count}(a, b)}{\text{count}(a)} = \frac{\text{count}(a, b)}{\sum_{b'} \text{count}(a, b')}$$
『こころ』の「先」なら：
$$P(\text{生} \mid \text{先}) = \frac{\text{「先生」の回数}}{\text{「先」の回数}} = \frac{597}{678} \approx 0.88$$

**⑦ Pythonの最小コードで検算**：
```python no-run
from collections import Counter
s = "あいうあいうあいあえ"
p = Counter(zip(s, s[1:]))
a_total = sum(n for (a, b), n in p.items() if a == "あ")   # 4
print("P(い|あ) =", p[("あ", "い")] / a_total)             # 0.75
print("P(え|あ) =", p[("あ", "え")] / a_total)             # 0.25
```

**⑧ よくある誤解**：$P(B \mid A)$ と $P(A \mid B)$ は**まったく別物**です。『こころ』では $P(\text{生} \mid \text{先}) = 597/678 \approx 0.88$ ですが、逆向きの $P(\text{先} \mid \text{生})$ は「生」の次に「先」が来る確率（たった4回）なのでほぼ $0$ です。縦棒 $\mid$ の**右側が条件（分母）**と意識しましょう。

**⑨ 言語モデルとの接続**：言語モデルとはまさに $P(\text{次の文字} \mid \text{それまでの文字})$ を出す仕組み（第3章）でした。今回は直前の**1文字だけ**を条件にして2つ（bi）の文字の組を数えるので、**bigram**（バイグラム）モデルと呼びます。

確認問題で、`zip` と条件付き確率の手計算・記号の読み方を定着させましょう！

#### 📝 確認問題（`ch04-q05` · Lv.2 · zip(text, text[1:]) による隣接ペア抽出と個数）

`zip(s, s[1:])` が作るペアの個数と中身を確認します。次のコードを実行すると何が表示されますか？

```python
s = "先生は"
pairs = list(zip(s, s[1:]))
print(len(pairs), pairs[0][0] + pairs[0][1])
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`zip(s, s[1:])` は元の文字列と1文字ずらした文字列を並べて、隣り合う2文字のペアを作ります。
- **ヒント 2**：長さ3の `"先生は"` からは `3 - 1 = 2` 個のペアが作られます。先頭のペア `pairs[0]` の2文字を連結しましょう。
- **期待される出力**：`2 先生`

**考え方**：`s = "先生は"`（長さ3）と `s[1:] = "生は"`（長さ2）を `zip` すると短い方の長さ2で止まり、`[('先', '生'), ('生', 'は')]` が作られます。

1. `s` = `"先生は"`、`s[1:]` = `"生は"`。
1. `list(zip(s, s[1:]))` = `[('先', '生'), ('生', 'は')]`。
1. `len(pairs)` は `2`、`pairs[0]` は `('先', '生')` なので連結すると `先生`。
1. 出力：`2 先生`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html#ex-ch04-q05

</details>

#### 📝 確認問題（`ch04-q06` · Lv.2 · 条件付き確率 P(b | a) の手計算）

『こころ』本文で「私」は（次の文字がある位置に）2697回出てきて、そのうち「私は」は1352回でした。条件付き確率 $P(\text{は} \mid \text{私})$ を小数第2位までで答えてください（誤差 $\pm 0.01$ まで正解）。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：条件付き確率 $P(\text{は} \mid \text{私})$ は「『私』が出た中で次が『は』だった割合」です。
- **ヒント 2**：分子に「私は」の回数 `1352`、分母に「私」の回数 `2697` を置いて割り算しましょう。
- **正解**：`0.5`

**考え方**：「私」が出たという条件のもとで次に「は」が来る割合は $1352 \div 2697 \approx 0.5013$、つまり約50%です。

1. 条件（分母）：「私」の回数 $= 2697$。
1. 条件を満たしかつ次が「は」（分子）：「私は」の回数 $= 1352$。
1. $1352 \div 2697 \approx 0.5013 \approx 0.50$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html#ex-ch04-q06

</details>

*(この節の追加問題 `ch04-q07`, `ch04-q08` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html で自動採点・復習できます)*

## 漱石に書かせる

数えた結果を使えば、文章が作れます。今の文字の「次の文字の回数」をそのまま重みにして、くじを引くのです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import random

following = {}                       # 「文字 → その次の文字の回数」の辞書
for a, b in zip(text, text[1:]):      # 隣り合う2文字 a, b を順番に
    if a not in following:            # a が初めて出てきたら
        following[a] = Counter()      #   a 用の空のカウンターを用意
    following[a][b] += 1              # 「a の次に b が来た」回数を1増やす

def generate(start, n=80):
    out = start
    ch = start[-1]                    # 書き出しの最後の1文字から続ける
    for _ in range(n):
        c = following[ch]
        ch = random.choices(list(c.keys()), weights=list(c.values()))[0]
        out += ch
    return out

random.seed(1)
print(generate("私"))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
私は腎臓の運ばなた。私のた後姿を帯の頁されはな時先生の方がなかられていと私のだろう済むしたに成功を心は私に宛てどこころにものだ私がら、多に私は自然とは午眠り、あ百
```

</details>

> **🐍 Python・PyTorchのポイント：Counter の「0回」と random.choices**
>
> `Counter` は、まだ数えていないものを「0回」として扱います。だから、初めて出てきた `b` でも、いきなり `following[a][b] += 1` と書けます（ふつうの辞書だと、ここで `KeyError` になります）。<br>
> `random.choices(候補のリスト, weights=重みのリスト)` は、重みに比例した確率で候補から1つ選びます（結果はリストで返るので `[0]` で取り出します）。第3章で自作した `sample` 関数と同じ仕事です。

**ボンゴ先生**：「私は腎臓の運ばなた。」……漱石先生に怒られそうだね。でも見てごらん。「私は」「先生の方が」「私に宛て」「自然とは」。**2文字だけ見れば**、どこも日本語として自然だ。このモデルは、直前の1文字しか覚えていない。文の前のほうで何を言っていたかは、きれいさっぱり忘れている。

> **🔬 やってみよう：いろいろな書き出しで**
>
> `random.seed(1)` の数字を変えたり、`generate("先")`、`generate("奥")`、`generate("Ｋ")`（『こころ』のKは全角の「Ｋ」です） のように書き出しを変えたりしてみましょう。毎回違う「漱石っぽい何か」が生まれます。気に入ったものがあったら、それがあなたのモデルの最初の作品です。

<details><summary><b>基礎の補足：理解のための最小コード：`random.choices` の入力・途中・出力</b>（クリックで開く）</summary>

| ステップ | 値（具体例） | 型 | 起きていること |
|---|---|---|---|
| 入力（候補） | `keys = ['い', 'え']` | `list` of `str` | `list(c.keys())` で候補を取り出す |
| 入力（重み） | `weights = [3, 1]` | `list` of `int` | `list(c.values())` で回数（合計1でなくてOK）を渡す |
| 内部処理 | 合計 `4` で割った確率 `[0.75, 0.25]` でくじ引き | 内部正規化 | 第3章の `sample` と同じ累積区間くじ引き |
| 出力 | `['い']` → `[0]` で `'い'` | `list` → `str` | 複数抽出（`k=...`）にも対応するため必ずリストで返るので `[0]` が必須 |

```python no-run
import random
keys = ["い", "え"]
weights = [3, 1]              # 3:1 の割合 → 75% と 25%
picked_list = random.choices(keys, weights=weights)
print(type(picked_list), picked_list, picked_list[0])   # <class 'list'> ['い'] い
```

</details>

#### 📝 確認問題（`ch04-q09` · Lv.2 · random.choices の重み付けと戻り値（リスト）の [0]）

`random.choices` の戻り値の型と、重み `weights` に0を混ぜたときの動きを確認します。次のコードを実行すると何が表示されますか？

```python
import random
res = random.choices(["生", "へ", "刻"], weights=[0, 5, 0])
print(type(res).__name__, len(res), res[0])
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`random.choices` は1個だけ選ぶときも必ず「リスト」を返します。また `weights` の中で0以外の候補に注目しましょう。
- **ヒント 2**：重み `[0, 5, 0]` なら必ず2番目の `"へ"` が選ばれ、`res` は長さ1のリスト `['へ']` になります。
- **期待される出力**：`list 1 へ`

**考え方**：`random.choices` は複数個選ぶこと（`k=...`）にも対応しているため、1個選ぶときも長さ1の `list` を返します。重み `[0, 5, 0]` は比率に自動正規化されて `[0.0, 1.0, 0.0]` になるため、必ず `"へ"` が選ばれます。

1. 重み `[0, 5, 0]` → 合計 `5` で割った確率 `[0.0, 1.0, 0.0]`。
1. `res` は長さ1のリスト `['へ']`。
1. `type(res).__name__` は `list`、`len(res)` は `1`、`res[0]` は `'へ'`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html#ex-ch04-q09

</details>

#### 📝 確認問題（`ch04-q10` · Lv.4 · 【自力実装】bigramの条件付き確率 P(b | target) を計算する関数）

文字列 `text` と条件となる1文字 `target` を受け取り、`target` の次に来る各文字 `b` に対する条件付き確率 $P(b \mid \text{target})$ を辞書 `{b: 確率}` として返す関数 `bigram_probs(text, target)` を自力実装してください（`target` の次に続く文字が1つもない場合は空辞書 `{}` を返す）。

**ひな形コード：**
```python
from collections import Counter

def bigram_probs(text, target):
    # 1. zip(text, text[1:]) を回し、a == target のときの b を Counter で数える
    # 2. 合計 total = sum(counts.values()) を求める
    # 3. total == 0 なら {} を返す
    # 4. {b: n / total for b, n in counts.items()} を返す
    pass
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`zip(text, text[1:])` で隣接ペア `(a, b)` を取り出し、`a == target` のときだけ `b` の回数を `Counter` で数えます。
- **ヒント 2**：合計 `total = sum(counts.values())` が `0` なら空辞書 `{}` を返し、そうでなければ各文字を `n / total` にした辞書を返します。

**解答例：**
```python
from collections import Counter

def bigram_probs(text, target):
    counts = Counter()
    for a, b in zip(text, text[1:]):
        if a == target:
            counts[b] += 1
    total = sum(counts.values())
    if total == 0:
        return {}
    return {b: n / total for b, n in counts.items()}
```

**検算テスト（Colabで実行して確認）：**
```python
p = bigram_probs("あいうあいうあいあえ", "あ")
assert abs(p["い"] - 0.75) < 1e-9
assert abs(p["え"] - 0.25) < 1e-9
assert abs(sum(p.values()) - 1.0) < 1e-9
assert bigram_probs("あいう", "う") == {}
assert bigram_probs("あいう", "ん") == {}
print("ALL PASSED")
```

**考え方**：条件 `a == target` で絞り込んだ `Counter` を作り、その合計 `total` で割ることで、定義式 $P(b \mid a) = \text{count}(a, b) / \text{count}(a)$ をそのままコードで表現できます。

1. `counts = Counter()` を用意し、`for a, b in zip(text, text[1:]):` で `a == target` なら `counts[b] += 1`。
1. `total = sum(counts.values())` を計算。`0` ならゼロ除算を防ぐため `{}` を返す。
1. `{b: n / total for b, n in counts.items()}` で各確率を辞書にして返す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html#ex-ch04-q10

</details>

## なぜ1文字では足りないのか

> **💡 理論のポイント：マルコフ性と n-gram**
>
> bigramモデルは「次の文字は、**直前の1文字だけ**で決まる」と仮定しています。この「直前だけを見る」性質を **マルコフ性** と言います（「マルコフ連鎖」という名前を聞いたことがあるかもしれません）。
>
> もっと前まで見れば、もっと自然になるはずです。直前2文字なら **trigram**（トライグラム）、一般に直前 $n-1$ 文字を見るモデルを **n-gram** モデルと呼びます。2010年代の半ばごろまで、機械翻訳や音声認識の中心はこのn-gramモデルでした。

では、直前2文字を見るtrigramを作ってみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
following3 = {}
for a, b, c in zip(text, text[1:], text[2:]):
    key = a + b
    if key not in following3:
        following3[key] = Counter()
    following3[key][c] += 1

def generate3(start, n=80):
    out = start
    for _ in range(n):
        c = following3.get(out[-2:])  # 最後の2文字の「次の文字の回数」。辞書になければ None
        if c is None:                 # 続きが見つからなければ
            break                     #   ループを途中で抜けて終わり
        out += random.choices(list(c.keys()), weights=list(c.values()))[0]
    return out

random.seed(1)
print(generate3("私は"))
print("2文字の組み合わせの種類：", len(following3))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
私はまるでした。先生を見詰めているらしませんからＫに向って、先生が不意にその他妹の夫だの、先生ははたしかし人間が来ると、知らないんでした。奥さんにはまた愛するといて
2文字の組み合わせの種類： 21562
```

</details>

「先生を見詰めている」「Ｋに向って」「奥さんにはまた愛する」……かなり『こころ』です！ 直前2文字を見るだけで、ぐっと文章らしくなりました。じゃあ、10文字見れば完璧では？

> **💡 理論のポイント：データが足りない問題（スパース性）**
>
> そうはいきません。見る文字数を増やすと、**組み合わせの数が爆発**します。語彙が2061文字なら、組み合わせの数は文字数に対して指数関数的に膨れ上がります。
>
> - 直前1文字（bigram）：$2061^2 \approx 425$万通り（『こころ』に出てくるのは約2万種類だけ）
> - 直前2文字（trigram）：$2061^3 \approx 87.5$億通り
> - 直前10文字（11-gram）：$2061^{11} \approx 2.85 \times 10^{36}$通り（地球上の砂粒の数 $\sim 10^{19}$〜$10^{20}$ を桁違いに上回る天文学的数字！ ちなみに観測可能な宇宙全体の原子数は約 $10^{80}$ 個で、さらに25-gramあたりでそれも超えます）
>
> 『こころ』全体でもたった16万字しかありません。文字数を少し増やしただけで、組み合わせの大部分はデータの中に**一度も出てこなくなります**。
>
> 一度も見たことがない並びに出会うと、数えるだけのモデルは「確率0」と答えるしかありません（上の `generate3` が `None` で止まるのは、このときです）。表のマス目の大半が0でスカスカになってしまうこの現象を、**スパース性**（まばらさ）の問題と言います。
>
> 人間なら、「先生は」を見たことがあれば「教授は」の次も似たような言葉が来そうだと推測できます。「先生」と「教授」が**似ている**ことを知っているからです。数えるだけのモデルには、この「似ている」が分かりません。
>
> この壁を越えるために、文字を「数の組（ベクトル）」で表して、似た文字を近くに置く方法を学びます。それがニューラルネットワークの出番であり、第9章の「埋め込み」です。

#### 📝 確認問題（`ch04-q11` · Lv.2 · マルコフ性と bigram / trigram / n-gram の関係）

直前の **2文字** を見て次の1文字を予測するモデル（例：$P(c \mid a, b)$）は何と呼ばれ、一般の $n$-gram モデルは何文字を条件にしますか？

- **a.** 直前2文字を見るのは **unigram（1-gram）** モデル。一般の $n$-gram は文全体に含まれる単語の総数を数えて予測する
- **b.** 直前2文字を見るモデルは有限の過去しか見ないためマルコフ性を持たず、$n$-gram モデルの仲間には含まれない
- **c.** 直前2文字を見るのは **trigram（3-gram）**。一般の $n$-gram は直前 **$n - 1$ 文字** を条件にする
- **d.** 直前2文字を見るのは **bigram（2-gram）** モデル。一般の $n$-gram は直前 **$n$ 文字** を条件にして次の1文字を予測する

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：$n$-gram の $n$ は「条件の文字数」ではなく「条件＋予測する次の1文字」の合計文字数です。
- **ヒント 2**：「条件2文字＋次の1文字＝合計3文字の組」なので 3-gram（trigram）、一般に $n$-gram の条件は $n-1$ 文字です。
- **正解**：`c`

**考え方**：bigram は `(a, b)` の2文字組（条件1文字）、trigram は `(a, b, c)` の3文字組（条件2文字）、一般に $n$-gram は直前 $n-1$ 文字を条件にします。

1. unigram ($n=1$)：条件0文字、$P(a)$。
1. bigram ($n=2$)：条件1文字、$P(b \mid a)$。
1. trigram ($n=3$)：条件2文字、$P(c \mid a, b)$。
1. $n$-gram：条件 $n-1$ 文字。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html#ex-ch04-q11

</details>

#### 📝 確認問題（`ch04-q12` · Lv.3 · n-gram の組み合わせ数 V^n の爆発を計算する）

語彙数 $V = 10$ 文字の小さな世界を考えます。直前 **3文字** を見て次の1文字を予測する **4-gram** モデルでは、理論上の4文字の組み合わせは何通りになりますか？（整数で入力）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：4文字の並びの各位置に、語彙数 $V = 10$ 通りの文字がそれぞれ独立に入ります。
- **ヒント 2**：組み合わせ総数は $V^n = 10^4$ です。10を4回掛けて整数で答えましょう。
- **正解**：`10000` 通り

**考え方**：語彙数 $V$ の $n$-gram の組み合わせ総数は $V^n$ です。$V = 10, n = 4$ なので $10^4 = 10000$ 通りになります。

1. 1文字目：10通り、2文字目：10通り、3文字目：10通り、4文字目：10通り。
1. 総数 $= 10^4 = 10000$ 通り。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html#ex-ch04-q12

</details>

*(この節の追加問題 `ch04-q13`, `ch04-q14` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html で自動採点・復習できます)*

## 確率の表として眺める

最後に、bigramモデルの正体を「表」として見ておきましょう。行が「今の文字」、列が「次の文字」、中身が回数です。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
look = ["先", "生", "私", "は", "。"]
print("今＼次 " + " ".join(f"{b:>4}" for b in look))
for a in look:
    row = " ".join(f"{pairs[(a, b)]:>5}" for b in look)
    print(f"  {a}   {row}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
今＼次    先    生    私    は    。
  先       0   597     0     2     0
  生       4     1     0   176     0
  私       0     0     0  1352     0
  は      58     6   145    11     1
  。     112     3   911     3     0
```

</details>

「私→は」が1352回、「。→私」が911回。文が終わると、次の文は「私」で始まることが多い。語り手の「私」の物語らしい表です。

本当の表は2061行×2061列、約420万マスもあります。この巨大な「数の表」を効率よく扱う道具が、第6章で学ぶ **行列** と **テンソル** です。

#### 📝 確認問題（`ch04-q15` · Lv.3 · bigram 遷移表（行＝今の文字、列＝次の文字）の読み取り）

本文の5文字の回数表（行＝今の文字、列＝次の文字）で、「`。` の行・`私` の列」が `911`、「`私` の行・`。` の列」が `0` でした。この表から正しく読み取れることはどれですか？

- **a.** 句点「。」の直後に「私」が来る回数は911回あるが、「私」の直後に句点「。」が来る回数は0回である
- **b.** 「私」で終わる文が911回ある一方で、句点「。」の直後に始まる文は0回であるという意味になる
- **c.** 遷移表は行と列を入れ替えても必ず同じ数値になるはずなので、911と0が一致しないのは集計バグである
- **d.** 本文全体における文字「私」の総出現回数が911回であり、「。」の総出現回数が0回である

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：表の見出し `今＼次` が「行＝今の文字、列＝次の文字」という向きを表している点に注目しましょう。
- **ヒント 2**：`(行=。, 列=私)` は「`。` → `私`」、`(行=私, 列=。)` は「`私` → `。`」の回数です。
- **正解**：`a`

**考え方**：遷移表では `(行 a, 列 b)` が `a → b` の回数（や確率 $P(b \mid a)$）を表します。そのため各**行**をその行の合計で割ると条件付き確率分布になります。

1. 行＝今の文字、列＝次の文字。
1. `(。, 私) = 911` → 文末「。」の次の文が「私」で始まることが911回。
1. `(私, 。) = 0` → 「私。」で終わる箇所は0回。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html#ex-ch04-q15

</details>

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **`UnicodeDecodeError`**：`decode("utf-8")` のままになっていませんか？ 青空文庫は `decode("cp932")` です。
> - **`IndexError: list index out of range`**（`split(…)[2]` の行）：ダウンロードしたファイルが本文ではなく、エラーページになっている可能性があります。`print(text[:300])` で中身を確認し、URLを見直しましょう。
> - **`KeyError`**（`following[ch]` の行）：書き出しの**最後の文字**が『こころ』に出てこない文字だと起きます。「私」「先生」など、本文にある文字で終わる書き出しにしましょう。

**✅ できたらチェック**

- [ ] 青空文庫のテキストを `cp932` で読み込めた
- [ ] 正規表現でルビを消せた（`?` の有無の違いも見た）
- [ ] `Counter` と `zip` で文字のペアを数えた
- [ ] bigramモデルで文章を生成した
- [ ] trigramにすると自然になるが、組み合わせが爆発することを説明できる

> **📝 まとめ**
>
> - 言語モデルの最も素朴な作り方は、「次の文字」を**数える**こと
> - 数えた割合が **条件付き確率** $P(\text{次} \mid \text{今})$。直前1文字なら **bigram**
> - 見る文字数を増やすと自然になるが、**組み合わせが爆発**してデータが足りなくなる
> - 機械学習では、データの**前処理**（文字コード・ルビ除去）が品質を左右する

**ボンゴ先生**：さて、bigramモデルは「なんとなく漱石」だった。でも「なんとなく」では研究にならない。モデルが**どれくらい**良いのか、点数をつける必要がある。次の章では、言語モデルの成績表のつけ方を学ぶ。そこで、確率を掛け算していたら、コンピュータが「0」と答えてしまう事件が起きる……。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html
- **次の章**：[第5章 モデルに点数をつける：損失とパープレキシティ](https://sakunokyumei.github.io/llm-research-textbook/05-loss.html)